# 🏙️ Modul 1: Ekstraksi Data Kualitas Udara & Cuaca DKI Jakarta (2024)

**Tujuan Notebook:**
1. Mengambil data historis kualitas udara Jakarta (PM2.5, PM10, AQI, CO, NO2) secara per jam (*hourly*) sepanjang tahun 2024 dari **Open-Meteo Air Quality API**.
2. Mengambil data cuaca Jakarta (Suhu, Kelembaban, Curah Hujan, Kecepatan Angin) untuk periode yang sama dari **Open-Meteo Archive Weather API**.
3. Menggabungkan (*merge*) kedua dataset berdasarkan stempel waktu (*timestamp*).
4. Menyimpan dataset mentah ke folder `data/raw/`.

### 1. Import Library yang Dibutuhkan

In [ ]:
import requests
import pandas as pd
import numpy as np
import os

# Pastikan folder data/raw dan data/processed tersedia
os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

print("Pandas version:", pd.__version__)

### 2. Parameter Lokasi & Periode Waktu
Kita menggunakan koordinat pusat DKI Jakarta:
- Latitude: `-6.2088`
- Longitude: `106.8456`
- Zona Waktu: `Asia/Jakarta` (UTC+7)
- Periode: 1 Januari 2024 - 31 Desember 2024 (1 tahun penuh = 8.784 jam)

In [ ]:
LATITUDE = -6.2088
LONGITUDE = 106.8456
START_DATE = "2024-01-01"
END_DATE = "2024-12-31"
TIMEZONE = "Asia/Bangkok"  # UTC+7 setara WIB

### 3. Ekstraksi Data Kualitas Udara (Air Quality)

In [ ]:
aq_url = "https://air-quality-api.open-meteo.com/v1/air-quality"
aq_params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "hourly": "pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,us_aqi",
    "timezone": TIMEZONE
}

print("Mengunduh data kualitas udara Jakarta 2024...")
response_aq = requests.get(aq_url, params=aq_params)
data_aq = response_aq.json()

# Konversi ke pandas DataFrame
df_aq = pd.DataFrame(data_aq["hourly"])
df_aq["time"] = pd.to_datetime(df_aq["time"])
print(f"Data kualitas udara berhasil diambil: {df_aq.shape[0]} baris, {df_aq.shape[1]} kolom")
df_aq.head()

### 4. Ekstraksi Data Cuaca Historis (Weather Archive)

In [ ]:
weather_url = "https://archive-api.open-meteo.com/v1/archive"
weather_params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "hourly": "temperature_2m,relative_humidity_2m,precipitation,rain,wind_speed_10m",
    "timezone": TIMEZONE
}

print("Mengunduh data cuaca Jakarta 2024...")
response_weather = requests.get(weather_url, params=weather_params)
data_weather = response_weather.json()

# Konversi ke pandas DataFrame
df_weather = pd.DataFrame(data_weather["hourly"])
df_weather["time"] = pd.to_datetime(df_weather["time"])
print(f"Data cuaca berhasil diambil: {df_weather.shape[0]} baris, {df_weather.shape[1]} kolom")
df_weather.head()

### 5. Penggabungan Dataset (Merge on Timestamp)

In [ ]:
df_combined = pd.merge(df_aq, df_weather, on="time", how="inner")

# Menambahkan fitur waktu turunan (feature engineering sederhana)
df_combined["date"] = df_combined["time"].dt.date
df_combined["hour"] = df_combined["time"].dt.hour
df_combined["day_name"] = df_combined["time"].dt.day_name()
df_combined["month"] = df_combined["time"].dt.month
df_combined["is_weekend"] = df_combined["time"].dt.dayofweek.isin([5, 6])

print("Ringkasan dataset gabungan:")
print(df_combined.info())
df_combined.head()

### 6. Simpan Dataset Mentah ke Disk
Kita simpan file ini ke `data/raw/jakarta_environment_hourly_2024.csv`.

In [ ]:
output_path = "../data/raw/jakarta_environment_hourly_2024.csv"
df_combined.to_csv(output_path, index=False)
print(f"Dataset berhasil disimpan ke: {output_path}")
print(f"Total baris: {len(df_combined):,}")